# SpaceX REST API collection

The project uses the documented public v4 past-launch endpoint, `https://api.spacexdata.com/v4/launches/past`. A GET request returns a JSON array with flight, rocket, payload, core, landing and outcome fields. A fixed IBM response is kept as a reproducible fallback if the live service is unavailable.

In [1]:
from pathlib import Path
import sys, json, pandas as pd, requests
root=Path.cwd()
if not (root/'src').exists(): root=root.parent
sys.path.insert(0,str(root/'src'))
from IPython.display import display
from collect_api import collect_static_course_json, normalize_records


## 1. Send a REST GET request and inspect the HTTP response

In [2]:
API_URL='https://api.spacexdata.com/v4/launches/past'
try:
    response=requests.get(API_URL,timeout=5)
    print('GET',API_URL)
    print('HTTP status:',response.status_code)
    response.raise_for_status()
    api_records=response.json()
    assert isinstance(api_records,list) and api_records
    records=api_records
    source='Live SpaceX API response'
    print('JSON records:',len(records))
except (requests.RequestException,ValueError,AssertionError) as err:
    print('Live endpoint unavailable in this execution environment:',type(err).__name__,str(err)[:180])
    records=collect_static_course_json()
    source='IBM archived API response; live request attempted, archived fixture used for reproducibility'
print('Data source used:',source)
print('First response keys:',sorted(records[0].keys()))

Live endpoint unavailable in this execution environment: ReadTimeout HTTPSConnectionPool(host='api.spacexdata.com', port=443): Read timed out. (read timeout=5)
Data source used: IBM archived API response; live request attempted, archived fixture used for reproducibility
First response keys: ['auto_update', 'capsules', 'cores', 'crew', 'date_local', 'date_precision', 'date_unix', 'date_utc', 'details', 'failures', 'fairings', 'flight_number', 'id', 'launchpad', 'links', 'name', 'net', 'payloads', 'rocket', 'ships', 'static_fire_date_unix', 'static_fire_date_utc', 'success', 'tbd', 'upcoming', 'window']


## 2. Normalize nested launch and core fields and inspect the outcome

In [3]:
frame=normalize_records(records)
out=root/'data/processed/api_launches_normalized.csv'
frame.to_csv(out,index=False)
print(f'Saved {len(frame)} records x {len(frame.columns)} fields to {out.relative_to(root)}')
display(frame.head(5))

Saved 107 records x 18 fields to data/processed/api_launches_normalized.csv


,flight_number,name,date_utc,upcoming,launch_success,rocket_id,launchpad_id,payload_ids,core_id,core_flight,gridfins,legs,reused,landing_attempt,landing_success,landing_type,landingpad_id,failures_json
0,1,FalconSat,2006-03-24T22:30:00.000Z,False,False,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86,"[""5eb0e4b5b6c3bb0006eeb1e1""]",5e9e289df35918033d3b2623,1,False,False,False,False,None,NaN,NaN,"[{""time"": 33, ""altitude"": null, ""reason"": ""mer..."
1,2,DemoSat,2007-03-21T01:10:00.000Z,False,False,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86,"[""5eb0e4b6b6c3bb0006eeb1e2""]",5e9e289ef35918416a3b2624,1,False,False,False,False,None,NaN,NaN,"[{""time"": 301, ""altitude"": 289, ""reason"": ""har..."
2,3,Trailblazer,2008-08-03T03:34:00.000Z,False,False,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86,"[""5eb0e4b6b6c3bb0006eeb1e3"", ""5eb0e4b6b6c3bb00...",5e9e289ef3591814873b2625,1,False,False,False,False,None,NaN,NaN,"[{""time"": 140, ""altitude"": 35, ""reason"": ""resi..."
3,4,RatSat,2008-09-28T23:15:00.000Z,False,True,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86,"[""5eb0e4b7b6c3bb0006eeb1e5""]",5e9e289ef3591855dc3b2626,1,False,False,False,False,None,NaN,NaN,[]
4,5,RazakSat,2009-07-13T03:35:00.000Z,False,True,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86,"[""5eb0e4b7b6c3bb0006eeb1e6""]",5e9e289ef359184f103b2627,1,False,False,False,False,None,NaN,NaN,[]


In [4]:
display(frame[['launch_success','landing_attempt','landing_success','landing_type']].value_counts(dropna=False).rename('records').reset_index().head(12))
print('The archived IBM response contains 107 records. A successful mission delivery (launch_success) is distinct from booster landing_success.')

,launch_success,landing_attempt,landing_success,landing_type,records
0,True,True,True,ASDS,45
1,True,False,NaN,NaN,26
2,True,True,True,RTLS,14
3,True,True,False,ASDS,9
4,True,True,True,Ocean,5
5,False,False,NaN,NaN,3
6,True,True,False,Ocean,2
7,False,True,NaN,ASDS,2
8,True,True,False,RTLS,1


The archived IBM response contains 107 records. A successful mission delivery (launch_success) is distinct from booster landing_success.
